# 17 — One-shot sealed-year evaluation (FY 2025-26)

Run on explicit user instruction, 2026-07-30. This notebook reads the sealed test year for the
**first and only** time. Everything in this cell is declared before any result exists; once scores
exist, nothing here changes. If execution fails before producing scores, the fix and re-run are
recorded in WORKLOG; there is no legitimate re-run after seeing numbers.

**Scope (matching the 17 spec pasted by the user):** the h=1 forecast layer — persistence, linear
delta (pooled + FedAvg), neural delta (pooled + FedAvg) — per specialty on the same public
`fl_client` pool as 14/16, every model scored on identical window-eligible test rows. The agent
simulation on the sealed year and band-coverage checks are **not** run here; each would be its own
explicit decision.

**Test rows:** windows whose origin month lies in FY 2025-26 — origins Apr 2025 – Feb 2026, targets
May 2025 – Mar 2026 (the mirror of validation's 11 origins). The Mar-2025-origin window (target
Apr 2025) belongs to neither training nor test and is excluded from both — asserted.

**Training, declared:**

- *Regime*: full history only. Decided on validation evidence before this run (full beats post-only
  at 17 of 18 C-specialties linear, 15 of 18 neural); the post-only variant is deliberately not run.
- *Linear, two cuts, both reported*: **val-frozen** (origins < Apr 2024 — byte-identical to the fits
  validation scored, the pure prospective replication) and **final** (every window whose target
  precedes Apr 2025 — the deployment refit; exact least squares, deterministic).
- *Neural, final cut only, seed 42*: architecture, budget and transforms verbatim from 16. A
  val-frozen neural "replication" is not run — cuDNN nondeterminism means it could not reproduce
  16's exact weights, and notebook 19 already brackets the seed variance (pooled ± 0.9, FedAvg
  ± 0.0 MAE at C_110).

**Metrics:** MAE + RMSE on levels, `r2_delta_secondary` on deltas (same definitions as 16), n beside
every score. Validation-year MAE (full regime, from `fl_neural_summary.csv`) is joined beside each
sealed score so the year-to-year comparison is in the stored output, not recomputed later.

**One deliberate code change from 16:** `build_windows` also returns each window's target month, so
the train/test split is asserted on target months directly (no training window may have a sealed
target) instead of being implied by a data cut. Everything else is copied verbatim.

Output: `data/processed/sealed_year_eval.csv`. The numbers are reported as-is — good or bad.

In [1]:
# Goal: window sets over the full 96 months, with the sealed/unsealed split proven by assert.
import numpy as np                           # window arrays, lstsq fits, arithmetic guards
import pandas as pd                          # panel loading and the result table
import torch                                 # the GRU clients (final training cut, seed 42)
from torch import nn                         # layers and the MSE training loss
import copy                                  # deepcopy hands each client its own copy of the global model
from pathlib import Path

PROC = Path("../data/processed")

TEST_START = pd.Timestamp("2025-04-01")      # the sealed year - read in this notebook for the only time
VAL_START = pd.Timestamp("2024-04-01")       # validation year FY 2024-25 (unchanged)

SEED = 42                                    # the project seed, unchanged
SEQ_LEN = 6                                  # every constant below is 16's, unchanged
HIDDEN = 32
ROUNDS, LOCAL_EPOCHS = 10, 5
POOLED_EPOCHS = ROUNDS * LOCAL_EPOCHS
LR = 1e-3

device = torch.device("cuda")                # CUDA verified on this machine; a missing GPU crashes loudly
panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
assert panel["month"].max() == pd.Timestamp("2026-03-01")   # the sealed year must actually be present
pool = panel[panel["fl_client"] & panel["is_public"]]        # same pool as 14/16 - no month cut this time
SPECIALTIES = sorted(pool["tfunc_code"].unique())

def build_windows(df):                       # verbatim from 16 except target_month, kept for the asserts
    df = df.sort_values(["provider_code", "month"]).copy()
    g = df.groupby("provider_code")["total_waiting"]
    df["target"] = g.shift(-1)               # next month's list
    df["target_month"] = df.groupby("provider_code")["month"].shift(-1)   # the month that target lives in
    df["delta_1"] = df["total_waiting"] - g.shift(1)
    df = df.dropna(subset=["target", "delta_1"])       # first row per provider has no momentum yet
    for c in ["pct_gt18", "pct_gt52"]:       # a zero list is 0/0 = NaN in the panel; an empty list
        df[c] = np.where(df["total_waiting"] > 0, df[c], 0.0)   # has no long-waiters, so the share is 0

    X, y, month, tmonth, prov, current, target, d1 = [], [], [], [], [], [], [], []
    for p, r in df.groupby("provider_code"):
        f = np.column_stack([np.log1p(r["total_waiting"]), r["delta_1"] / 100,
                             r["pct_gt18"], r["pct_gt52"]]).astype(np.float32)
        t = ((r["target"] - r["total_waiting"]) / 100).to_numpy(np.float32)
        for i in range(SEQ_LEN - 1, len(r)):           # a provider shorter than SEQ_LEN adds nothing
            X.append(f[i - SEQ_LEN + 1:i + 1]); y.append(t[i])
        month += list(r["month"].iloc[SEQ_LEN - 1:]); tmonth += list(r["target_month"].iloc[SEQ_LEN - 1:])
        prov += [p] * (len(r) - SEQ_LEN + 1)
        current += list(r["total_waiting"].iloc[SEQ_LEN - 1:])
        target += list(r["target"].iloc[SEQ_LEN - 1:]); d1 += list(r["delta_1"].iloc[SEQ_LEN - 1:])
    return (np.stack(X), np.array(y, dtype=np.float32), pd.Series(month), pd.Series(tmonth),
            np.array(prov), np.array(current, dtype=float), np.array(target, dtype=float),
            np.array(d1, dtype=float))

print(len(SPECIALTIES), "specialties,", f"{len(pool):,}", "pool rows over",
      pool["month"].nunique(), "months (sealed year included, deliberately, for the one-shot)")

24 specialties, 215,940 pool rows over 96 months (sealed year included, deliberately, for the one-shot)


In [2]:
# Goal: the client model and training procedures - verbatim from 16 - plus 11's exact linear fit.
class DeltaGRU(nn.Module):                   # one GRU layer reads the 6 months; a linear head -> delta/100
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(4, HIDDEN, batch_first=True)
        self.head = nn.Linear(HIDDEN, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1]).squeeze(-1)      # the last month's hidden state carries the sequence

def train_steps(model, X, y, epochs):        # full-batch Adam: client datasets are tiny (<= ~100 windows)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    mse = nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad(); loss = mse(model(X), y); loss.backward(); opt.step()

def fit_fedavg(client_data):                 # classic FedAvg: broadcast, local steps, weight-average
    glob = DeltaGRU().to(device)
    for _ in range(ROUNDS):
        states, ns = [], []
        for Xc, yc in client_data:
            local = copy.deepcopy(glob)
            train_steps(local, Xc, yc, LOCAL_EPOCHS)
            states.append(local.state_dict()); ns.append(len(yc))
        n_tot = sum(ns)
        glob.load_state_dict({k: sum(sd[k] * n for sd, n in zip(states, ns)) / n_tot
                              for k in states[0]})    # sample-weighted average of every weight tensor
    return glob

def fit_pooled(X, y):                        # centralised reference on the identical total step budget
    model = DeltaGRU().to(device)
    train_steps(model, X, y, POOLED_EPOCHS)
    return model

def fit_lin(d1_arr, ch_arr):                 # change = a + b * momentum, exact least squares (11's model)
    A = np.column_stack([np.ones(len(d1_arr)), d1_arr])
    beta, *_ = np.linalg.lstsq(A, ch_arr, rcond=None)
    return beta

def fed_lin(d1_arr, ch_arr, pr_arr):         # sample-weighted coefficient average across clients (11/14/16)
    bd = {p: (fit_lin(d1_arr[pr_arr == p], ch_arr[pr_arr == p]), int((pr_arr == p).sum()))
          for p in np.unique(pr_arr)}
    n_tot = sum(n for _, n in bd.values())
    return sum(b * n for b, n in bd.values()) / n_tot

In [3]:
# Goal: the one-shot scores - every model on the identical FY 2025-26 test rows, per specialty.
results, skipped = [], []
for tf in SPECIALTIES:
    X, y, month, tmonth, prov, current, target, d1 = build_windows(pool[pool["tfunc_code"] == tf])
    train_val = (month < VAL_START).to_numpy()         # 16's exact training windows, unchanged history
    train_fin = (tmonth < TEST_START).to_numpy()       # every window whose target is still unsealed
    test = (month >= TEST_START).to_numpy()            # origins Apr 2025 - Feb 2026, targets in FY 2025-26
    assert not (train_fin & test).any()                # the Mar-2025-origin boundary row is in neither set

    if tf == "C_110":                        # replication anchor: this construction must reproduce 16's
        val_rep = (month >= VAL_START).to_numpy() & (tmonth < TEST_START).to_numpy()
        assert int(val_rep.sum()) == 1401    # 16's eligible validation rows - a drifted window build trips this
        assert round(float(np.abs(target[val_rep] - current[val_rep]).mean()), 1) == 154.1
        assert month[test].nunique() == 11   # the mirror of validation's 11 origin months

    if int(test.sum()) == 0:                 # a specialty with no sealed-year windows in this pool
        skipped.append(tf)
        continue
    truth, cur, d1v = target[test], current[test], d1[test]
    dtrue = truth - cur
    ss_tot = ((dtrue - dtrue.mean()) ** 2).sum()
    Xv = torch.tensor(X[test]).to(device)
    n_test = int(test.sum())
    def score(pred):                         # MAE + RMSE on levels; R2 on the deltas (secondary) - 16's exact
        err = truth - pred
        r2 = 1 - ((err ** 2).sum() / ss_tot)
        return float(np.abs(err).mean()), float(np.sqrt((err ** 2).mean())), float(r2)
    def add(model, cut, pred):
        mae, rmse, r2 = score(pred)
        results.append({"tfunc_code": tf, "model": model, "train_cut": cut, "mae": mae,
                        "rmse": rmse, "r2_delta_secondary": r2, "n": n_test})

    add("persistence", "-", cur)
    for cut, in_tr in [("val_frozen", train_val), ("final", train_fin)]:
        bpool = fit_lin(d1[in_tr], target[in_tr] - current[in_tr])
        bfed = fed_lin(d1[in_tr], target[in_tr] - current[in_tr], prov[in_tr])
        add("linear_pooled", cut, cur + bpool[0] + bpool[1] * d1v)
        add("linear_fedavg", cut, cur + bfed[0] + bfed[1] * d1v)

    Xt = torch.tensor(X[train_fin]).to(device); yt = torch.tensor(y[train_fin]).to(device)
    pr = prov[train_fin]
    cd = [(Xt[pr == p], yt[pr == p]) for p in np.unique(pr)]
    torch.manual_seed(SEED)                  # identical init per model, exactly as 16 seeds its models
    npool = fit_pooled(Xt, yt)
    torch.manual_seed(SEED)
    nfed = fit_fedavg(cd)
    with torch.no_grad():                    # x100 undoes the declared target scaling
        add("neural_pooled", "final", cur + 100 * npool(Xv).cpu().numpy())
        add("neural_fedavg", "final", cur + 100 * nfed(Xv).cpu().numpy())
    print(f"{tf}: {n_test} test rows, {pd.Series(prov[test]).nunique()} providers "
          f"| sealed persistence MAE {results[-7]['mae']:,.1f} (n={n_test})")
print("skipped (zero sealed-year windows):", skipped)

C:\Users\heave\Documents\NHS UK FL+Agentic AI\nhs_FL_agentic\Lib\site-packages\torch\nn\modules\rnn.py:1449: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  result = _VF.gru(


C_100: 1298 test rows, 119 providers | sealed persistence MAE 114.1 (n=1298)


C:\Users\heave\Documents\NHS UK FL+Agentic AI\nhs_FL_agentic\Lib\site-packages\torch\nn\modules\rnn.py:1449: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  result = _VF.gru(


C_101: 1286 test rows, 117 providers | sealed persistence MAE 105.0 (n=1286)


C:\Users\heave\Documents\NHS UK FL+Agentic AI\nhs_FL_agentic\Lib\site-packages\torch\nn\modules\rnn.py:1449: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  result = _VF.gru(


C_110: 1407 test rows, 128 providers | sealed persistence MAE 151.5 (n=1407)


... (36 outputs left out)

C:\Users\heave\Documents\NHS UK FL+Agentic AI\nhs_FL_agentic\Lib\site-packages\torch\nn\modules\rnn.py:1449: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  result = _VF.gru(


X05: 1301 test rows, 120 providers | sealed persistence MAE 149.4 (n=1301)


C:\Users\heave\Documents\NHS UK FL+Agentic AI\nhs_FL_agentic\Lib\site-packages\torch\nn\modules\rnn.py:1449: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  result = _VF.gru(


X06: 1327 test rows, 124 providers | sealed persistence MAE 89.9 (n=1327)
skipped (zero sealed-year windows): ['X01']


In [4]:
# Goal: sealed_year_eval.csv with the validation-year MAE joined beside every sealed score, plus the verdicts.
res = pd.DataFrame(results)
valref = pd.read_csv(PROC / "fl_neural_summary.csv")   # the stored validation scores (full regime)
vmae = valref[valref["regime"].isin(["-", "full"])].set_index(["tfunc_code", "model"])["mae"]
res["val_mae_full"] = [vmae.get((r.tfunc_code, r.model), np.nan) for r in res.itertuples()]
# NaN val_mae_full marks the one validation gap: X01 was unscored in 16 (zero eligible windows there)
res.to_csv(PROC / "sealed_year_eval.csv", index=False)

pers = res[res["model"] == "persistence"].set_index("tfunc_code")["mae"]
print("=== C_110, FY 2025-26 (the anchor, one shot) ===")
print(res[res["tfunc_code"] == "C_110"].sort_values("mae")
      .round(1).to_string(index=False))

print()
for model in ["linear_fedavg", "neural_pooled", "neural_fedavg"]:
    for cut in (["val_frozen", "final"] if model.startswith("linear") else ["final"]):
        s = res[(res["model"] == model) & (res["train_cut"] == cut)].set_index("tfunc_code")["mae"]
        beats = int((s < pers.loc[s.index]).sum())
        print(f"{model:14s} [{cut:10s}] beats persistence at {beats} of {len(s)} specialties (sealed year)")

print()
drift = (pers / res[res["model"] == "persistence"].set_index("tfunc_code")["val_mae_full"] - 1) * 100
print("persistence MAE change vs validation year, per specialty (%):")
print(drift.dropna().sort_values().round(1).to_string())
print(res.shape, "->", PROC / "sealed_year_eval.csv")

=== C_110, FY 2025-26 (the anchor, one shot) ===
tfunc_code         model  train_cut   mae  rmse  r2_delta_secondary    n  val_mae_full
     C_110   persistence          - 151.5 251.5                -0.0 1407         154.1
     C_110 neural_pooled      final 151.6 252.0                -0.0 1407         150.6
     C_110 neural_fedavg      final 152.1 252.1                -0.0 1407         152.4
     C_110 linear_fedavg      final 153.0 254.7                -0.0 1407         154.7
     C_110 linear_pooled      final 153.0 253.8                -0.0 1407         154.4
     C_110 linear_pooled val_frozen 153.3 254.3                -0.0 1407         154.4
     C_110 linear_fedavg val_frozen 153.3 255.4                -0.0 1407         154.7

linear_fedavg  [val_frozen] beats persistence at 10 of 23 specialties (sealed year)
linear_fedavg  [final     ] beats persistence at 14 of 23 specialties (sealed year)
neural_pooled  [final     ] beats persistence at 15 of 23 specialties (sealed year)
ne